## 1.Current Situation 
Years ago, the customer identification rate (ITR/CRM/loyalty program registration) for customers shopping in stores was very low. It was impossible to calculate a detailed market share without knowing which neighborhood the customers were coming from.

Old Solution (Spending Survey): To address this blind spot, customers were manually asked which neighborhood they were from in 1 out of every 5 transactions at the checkout (a fixed rate of 20%).

Current Success (Bursa Marka Example): In 2025, the ITR rate at the Bursa Marka store reached 66%. Thanks to this automation, address data coverage per order increased from 18% to 32%, providing the store with an annual operational gain of +187 hours (~21 days).


## 2. Question: Do We Still Need Surveys?
With 66% ITR (32% address coverage in Marka Store) saving us 187 hours.

In [0]:
%sql
WITH base_sales AS (
    SELECT 
        sal.economical_businessunit_id AS store_code, -- Mağaza kodunu ekledik
        sal.transaction_id,
        mem.member_id,
        CASE WHEN mem.member_id IS NOT NULL THEN 1 ELSE 0 END as is_identified
    FROM prod_datalake_gold_sales.sales_detail sal
    LEFT JOIN prod_datalake_silver.f_member_purchase mem 
        ON mem.transaction_id = sal.transaction_id
    WHERE sal.year = 2025
    AND sal.economical_businessunit_country_code = 'TR'
      AND sal.transaction_channel_type = 'offline'
      AND sal.item_operation_type IN ('sale', 'return')
    GROUP BY 1, 2, 3, 4
),
address_validation AS (
    SELECT 
        b.store_code,
        b.transaction_id,
        b.is_identified,
        CASE WHEN ca.member_id IS NOT NULL THEN 1 ELSE 0 END as has_address_record, 
        CASE 
            WHEN ca.province IS NOT NULL 
                 AND ca.district IS NOT NULL 
                 AND ca.locality IS NOT NULL 
                 AND LENGTH(ca.locality) > 2 
            THEN 1 ELSE 0 
        END as is_clean_location
    FROM base_sales b
    LEFT JOIN prod_datalake_silver_customer_growth.d_member_address ca 
        ON ca.member_id = b.member_id 
        AND ca.address_type = 'BILLING' 
        AND ca.location = 'TR'
)
SELECT 
    store_code,
    COUNT(transaction_id) as total_transactions,
    
    -- ITR Performansı
    SUM(is_identified) as identified_transactions, 
    ROUND(100.0 * SUM(is_identified) / COUNT(transaction_id), 2) as ITR_Rate_Pct, 
    
    -- Temiz Coğrafi Veri Temsiliyeti
    SUM(CASE WHEN is_identified = 1 THEN is_clean_location ELSE 0 END) as verified_geo_transactions,
    ROUND(100.0 * SUM(CASE WHEN is_identified = 1 THEN is_clean_location ELSE 0 END) / COUNT(transaction_id), 2) as Representation_Rate_Pct,
    
    -- Stratejik Aksiyon Kararı (Yeni Akıllı Modelimiz)
    CASE 
        WHEN ROUND(100.0 * SUM(CASE WHEN is_identified = 1 THEN is_clean_location ELSE 0 END) / COUNT(transaction_id), 2) >= 80.0 
        THEN 'ANKETI TAMAMEN KAPAT (CRM Yeterli)'
        WHEN ROUND(100.0 * SUM(CASE WHEN is_identified = 1 THEN is_clean_location ELSE 0 END) / COUNT(transaction_id), 2) BETWEEN 50.0 AND 79.99
        THEN 'SEZONSAL MINI KOTA (Yılda 2 Kez)'
        ELSE 'AYLIK SABIT KOTA (Aylık 368 Anket)'
    END AS strategic_survey_action
FROM address_validation
GROUP BY store_code
ORDER BY Representation_Rate_Pct DESC;

Databricks visualization. Run in Databricks to view.

## Answer: Yes.

Database analyses of all our stores across Turkey have revealed that our ITR-based automated address representation rate is at a maximum of 38%. This data definitively proves that we still need our Spending Survey operation to maintain the accuracy of our geographical market share calculations.

We must continue with the "Data Fusion" model. However, assuming that the geographic distribution of the remaining 62% anonymous audience is exactly the same as our 32% loyal audience is a statistical gamble. While loyal customers using the application come from distant districts, anonymous customers might only be residents of the neighborhood near that shopping mall.

Survey Abandonment Limit (80% Rule): If our automatic address data coverage exceeded 80%, we could completely discard the survey. Since we are currently at 32%, we need a smart survey model to close the gap.

## New Model:
However, to lighten the operational burden at cash registers, the old "1 survey for every 5 transactions" (20% fixed rate) rule has been completely abandoned. Instead, the Cochran Model, which provides a statistically significant 95% confidence level, has been implemented.

## Cochran Model Calculation for Optimal Survey Amount
### Descriptions
- Population (N): Monthly anonymous nb of transaction, store base
- Sample (n): Monthly, store base

Till type, Gender, Age, Day of the week should be represented equally

In 2025, max daily survey amount for a store is **234** and min is **33**; in 2026 max is **233** and min is **0.14**.

In [0]:
import math

def calculate_sample_size(confidence_level=0.95, margin_of_error=0.05, p=0.5, population=None):
    """
    confidence_level: Confidence level (e.g., 0.95)
    margin_of_error: Margin of error (e.g., 0.05)
    p: Population heterogeneity / proportion (0.5 if unknown)
    population: Population size (N). If None, calculates for an infinite population.
    """
    # 1. Determine the Z-score based on the confidence level (Most common: 95% and 99%)
    if confidence_level == 0.95:
        z_score = 1.96
    elif confidence_level == 0.99:
        z_score = 2.58
    else:
        # General Z-score calculation (Standard Normal Distribution)
        import scipy.stats as stats
        z_score = stats.norm.ppf(1 - (1 - confidence_level) / 2)
        
    # 2. Cochran's Infinite Population Formula (n0)
    # n0 = (Z^2 * p * (1-p)) / e^2
    n0 = (pow(z_score, 2) * p * (1 - p)) / pow(margin_of_error, 2)
    
    # 3. If population (N) is not specified or infinite, return n0 directly (The magic number 385!)
    if population is None:
        return math.ceil(n0)
    
    # 4. If population is finite/known, apply the Finite Population Correction (FPC)
    n = n0 / (1 + ((n0 - 1) / population))
    return math.ceil(n)

# ---- SIMULATION ----
print(f"Required sample size for an Infinite (Very Large) Population: {calculate_sample_size()} people (Magic Number 385)")
print("-" * 65)

# Let's observe how the required sample size plateaus at 385 as the population grows:
test_populations = [500, 1000, 5000, 10000, 100000, 1000000, 10000000]

print(f"{'Population Size (N)':<25} | {'Required Sample Size (n)':<20}")
print("-" * 65)
for N in test_populations:
    sample_size = calculate_sample_size(population=N)
    print(f"{N:<25,} | {sample_size:<20}")

In [0]:
import math

def calculate_transition_steps(confidence_level=0.95, margin_of_error=0.05, micro_share=0.005):
    print("STEP-BY-STEP MATHEMATICAL TRANSITION: FROM 385 TO 750\n" + "="*55)
    
    # STEP 1: Cochran's Infinite Population Formula
    # Formula: n0 = (Z^2 * p * (1-p)) / e^2
    z_score = 1.96  # For 95% Confidence
    p_heterogeneity = 0.50  # Maximum variance
    
    n_cochran = (pow(z_score, 2) * p_heterogeneity * (1 - p_heterogeneity)) / pow(margin_of_error, 2)
    final_cochran = math.ceil(n_cochran)
    print(f"Step 1: General Store-Level Target (Cochran)  = {final_cochran} surveys/month")
    print(f"        -> Formula: ({z_score}^2 * {p_heterogeneity} * {1-p_heterogeneity}) / {margin_of_error}^2")
    print("-" * 55)
    
    # STEP 2: Micro-Neighborhood Level (Binomial Probability)
    # We want P(X >= 1) >= 0.95  =>  1 - (1 - p)^n <= 0.05
    # Logarithmic solution: n >= ln(0.05) / ln(1 - p)
    target_risk = 1 - confidence_level  # 0.05 risk allowed
    
    n_binomial = math.log(target_risk) / math.log(1 - micro_share)
    final_binomial = math.ceil(n_binomial)
    print(f"Step 2: Micro-Neighborhood 0.5% Target (Binom) = {final_binomial} surveys/month")
    print(f"        -> Formula: ln({target_risk}) / ln(1 - {micro_share})")
    print("-" * 55)
    
    # STEP 3: Operational Safety Buffer (Final Decision)
    # 598 is rounded up to 750 to provide ~25% buffer for bad/empty survey data
    final_target = 750
    daily_target = final_target / 30
    
    # Calculate the actual capture probability at 750 surveys
    prob_at_750 = (1 - pow(1 - micro_share, final_target)) * 100
    
    print(f"Step 3: Final Operational Target with Buffer    = {final_target} surveys/month")
    print(f"        -> Daily Target: {final_target} / 30 days = {int(daily_target)} surveys/day")
    print(f"        -> Actual Capture Probability at 750    = {prob_at_750:.2f}%")
    print("="*55)

# Run the transition calculation
calculate_transition_steps()

In [0]:
%sql
WITH main AS (
    -- Sizin orijinal sorgunuz: 2025 yılı fiziki mağaza işlem özetleri
    SELECT 
        CONCAT(sop.but_num_business_unit, ' ', sop.store_name) AS store_name, 
        CASE WHEN mem.member_id IS NULL THEN 'non_identified' ELSE 'identified' END AS is_identified_transaction, 
        COUNT(DISTINCT sal.transaction_id) AS nb_order 
    FROM datalake_gold.sales.sales_detail AS sal 
    LEFT JOIN datalake_silver.member_purchase.f_member_purchase AS mem 
        ON mem.transaction_id = sal.transaction_id 
    LEFT JOIN hive_metastore.dev_country_turkey.store_openings AS sop 
        ON sop.but_code_international = sal.merchant_businessunit_gln 
    WHERE sal.year = 2025
        AND sal.economical_businessunit_country_code = 'TR' 
        AND sal.transaction_channel_type = 'offline' 
        AND sal.merchant_businessunit_id <> '7-2991-2991' 
        AND sal.merchant_businessunit_id <> '7-2868-2868' 
        AND sal.workstation_id <> 20 
    GROUP BY all
),
store_monthly_base AS (
    -- 2. ADIM: Yıllık veriyi aylık ortalamaya (N) çeviriyoruz ve mağaza bazlı tanımlılık oranını buluyoruz
    SELECT 
        store_name,
        -- Mağazanın tanımlı işlem oranı % (Anket muafiyeti kontrolü için)
        ROUND(SUM(CASE WHEN is_identified_transaction = 'identified' THEN nb_order ELSE 0 END) / NULLIF(SUM(nb_order), 0) * 100, 1) AS identification_rate_pct,
        -- Aylık Ortalama Anonim İşlem Hacmi (Bizim Ana Kütlemiz: N)
        ROUND(SUM(CASE WHEN is_identified_transaction = 'non_identified' THEN nb_order ELSE 0 END) / 12.0, 0) AS N_monthly_anonim_order
    FROM 
        main 
    WHERE 
        store_name IS NOT NULL
    GROUP BY 
        store_name
),
sample_size_calculation AS (
    -- 3. ADIM: Cochran Sonlu Popülasyon formülünü uyguluyoruz
    -- %95 Güven, %5 Hata Payı ve p=0.5 için baz sayı n0 = 384.16
    SELECT 
        store_name,
        identification_rate_pct,
        N_monthly_anonim_order,
        ROUND(
            384.16 / (1.0 + (383.16 / NULLIF(N_monthly_anonim_order, 0))), 
            0
        ) AS required_monthly_sample_size
    FROM 
        store_monthly_base
)
-- 4. ADIM: Aylık hedefi operasyonel kotalara (Günlük, Hafta İçi, Hafta Sonu) bölüyoruz
SELECT 
    store_name AS store_name,
    identification_rate_pct AS identification_rate_pct,
    N_monthly_anonim_order AS monthly_avg_anon_orders,
    required_monthly_sample_size AS required_monthly_survey_target,
    
    -- Emniyet payı dahil düz günlük hedef (Ayda 30 gün üzerinden)
    CEILING((required_monthly_sample_size + 15) / 30.0) AS daily_flat_target,
    
    -- Operasyonel rahatlık için optimize edilmiş kotalar (Ayda 22 hafta içi, 8 hafta sonu günü varsayımıyla)
    -- Yükün %75'ini sakin günlere (hafta içi), %25'ini yoğun günlere (hafta sonu) dağıtır
    CEILING((required_monthly_sample_size * 0.75) / 22.0) AS weekday_daily_target,
    CEILING((required_monthly_sample_size * 0.25) / 8.0) AS weekend_daily_target
FROM 
    sample_size_calculation
ORDER BY 
    monthly_avg_anon_orders DESC;

Databricks visualization. Run in Databricks to view.

In the market, there are micro-neighborhoods that we call "Long Tail," located very far from Decathlon stores or with very small populations, contributing only 0.5% of the total turnover.

This code used the Binomial Distribution formula to answer the following question:

"If I collect X number of surveys per month, what is the probability that I will encounter at least 1 or 2 people from that smallest neighborhood?"

The output of the code gave us the following risk table:

If you conduct 13 surveys per day (380 per month): Your chance of catching at least 1 person from that small neighborhood is 85%, and your chance of catching at least 2 people is 56%. In other words, there was a risk of not seeing that neighborhood on the map.

If you conducted 25 surveys a day (750 a month): your chance of identifying at least one person from that small neighborhood jumped to 97.67%, and your chance of identifying at least two people jumped to 88%. The English sentence at the end of the code summarized this as "750 surveys guarantee you'll see even the smallest neighborhoods on the map."

In [0]:
import math

def calculate_neighborhood_capture_probability(monthly_surveys=750, neighborhood_share=0.005):
    """
    Calculates the probability of capturing at least 1 and at least 2 customers 
    from a specific micro-neighborhood using the Binomial Distribution.
    
    monthly_surveys (n): Total surveys conducted in a store per month.
    neighborhood_share (p): The market share of the micro-neighborhood (e.g., 0.005 for 0.5%).
    """
    # 1. Probability of NOT capturing anyone from this neighborhood (0 hits)
    # P(X = 0) = (1 - p)^n
    prob_zero_hits = pow(1 - neighborhood_share, monthly_surveys)
    
    # 2. Probability of capturing exactly 1 person
    # P(X = 1) = n * p^1 * (1 - p)^(n-1)
    prob_one_hit = monthly_surveys * neighborhood_share * pow(1 - neighborhood_share, monthly_surveys - 1)
    
    # 3. Probability of capturing AT LEAST 1 person
    # P(X >= 1) = 1 - P(X = 0)
    prob_at_least_one = 1 - prob_zero_hits
    
    # 4. Probability of capturing AT LEAST 2 people (for better data validation)
    # P(X >= 2) = 1 - [P(X = 0) + P(X = 1)]
    prob_at_least_two = 1 - (prob_zero_hits + prob_one_hit)
    
    return prob_at_least_one * 100, prob_at_least_two * 100

# ---- SIMULATION ----
print("Neighborhood Capture Probability Simulation (Long Tail Analysis)")
print("-" * 75)

# Testing our two main scenarios: 15 surveys/day (380/mo) vs. 25 surveys/day (750/mo)
survey_scenarios = [380, 750, 1500] 
target_share = 0.005 # 0.5% micro-neighborhood share

print(f"Targeting a Micro-Neighborhood with a {target_share*100}% market share:")
print(f"{'Daily Target':<15} | {'Monthly Surveys (n)':<20} | {'Prob. of At Least 1 Hit':<25} | {'Prob. of At Least 2 Hits'}")
print("-" * 75)

for surveys in survey_scenarios:
    daily_target = round(surveys / 30)
    p1, p2 = calculate_neighborhood_capture_probability(monthly_surveys=surveys, neighborhood_share=target_share)
    print(f"~{daily_target:<13} | {surveys:<20,} | {p1:<23.2f}% | {p2:.2f}%")

print("-" * 75)
print("Conclusion: 25 surveys/day (750/mo) gives you a solid 97.67% guarantee to capture\n"
      "even the smallest neighborhoods (0.5% share), ensuring store managers see them on the map.")

In [0]:
%sql
WITH orijinal_dagilim AS (
    -- 1. ADIM: 2025 yılındaki 2.2 milyon verinin orijinal ilçe bazlı yüzdelerini hesaplıyoruz
    SELECT 
        province,
        district,
        COUNT(spending) AS orijinal_anket_adedi,
        (COUNT(spending) / SUM(COUNT(spending)) OVER()) * 100 AS orijinal_pay_yuzde
    FROM dev_country_turkey.spendingapp
    WHERE date_format(to_timestamp(survey_date, 'M/d/yyyy h:m:ss a'), 'yyyy') = '2025'
      AND store_code = '1619'
    GROUP BY province, district
),
rastgele_orneklem AS (
    -- 2. ADIM: Ham veriden TAMAMEN RASTGELE sadece 368 adet anket satırı seçiyoruz (Yeni model simülasyonu)
    SELECT 
        province,
        district
    FROM dev_country_turkey.spendingapp
    WHERE date_format(to_timestamp(survey_date, 'M/d/yyyy h:m:ss a'), 'yyyy') = '2025'
      AND store_code = '1619'
    ORDER BY rand() -- Veriyi tamamen karıştırır
    LIMIT 368       -- İçinden sadece 368 tane çeker
),
yeni_dagilim AS (
    -- 3. ADIM: Bu seçtiğimiz 368 adet rastgele verinin ilçe bazlı yüzdelerini hesaplıyoruz
    SELECT 
        province,
        district,
        COUNT(*) AS yeni_sistem_anket_adedi,
        (COUNT(*) / 368) * 100 AS yeni_sistem_pay_yuzde
    FROM rastgele_orneklem
    GROUP BY province, district
)
-- 4. ADIM: İki sonuç kümesini yan yana getirip farkı (ispatı) görüyoruz
SELECT 
    o.province,
    o.district,
    o.orijinal_anket_adedi,
    ROUND(o.orijinal_pay_yuzde, 2) AS orijinal_pay_yuzde,
    COALESCE(y.yeni_sistem_anket_adedi, 0) AS yeni_sistem_anket_adedi,
    ROUND(COALESCE(y.yeni_sistem_pay_yuzde, 0), 2) AS yeni_sistem_pay_yuzde,
    ROUND(COALESCE(y.yeni_sistem_pay_yuzde, 0) - o.orijinal_pay_yuzde, 2) AS fark_yuzde_puani
FROM orijinal_dagilim o
LEFT JOIN yeni_dagilim y 
  ON o.province = y.province AND o.district = y.district
ORDER BY o.orijinal_anket_adedi DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH orijinal_dagilim AS (
    -- 1. ADIM: 2025 yılındaki 2.2 milyon verinin MAHALLE bazlı orijinal yüzdelerini hesaplıyoruz
    SELECT 
        province,
        district,
        neighborhood, -- Eğer tabloundaki kolon adı 'mahalle' ise burayı değiştirin
        COUNT(spending) AS orijinal_anket_adedi,
        (COUNT(spending) / SUM(COUNT(spending)) OVER()) * 100 AS orijinal_pay_yuzde
    FROM dev_country_turkey.spendingapp
    WHERE date_format(to_timestamp(survey_date, 'M/d/yyyy h:m:ss a'), 'yyyy') = '2025'
      AND store_code = '1619'
    GROUP BY province, district, neighborhood
),
rastgele_orneklem AS (
    -- 2. ADIM: Ham veriden TAMAMEN RASTGELE sadece 368 adet anket satırı seçiyoruz
    SELECT 
        province,
        district,
        neighborhood
    FROM dev_country_turkey.spendingapp
    WHERE date_format(to_timestamp(survey_date, 'M/d/yyyy h:m:ss a'), 'yyyy') = '2025'
      AND store_code = '1619'
    ORDER BY rand() -- Veriyi tamamen karıştırır
    LIMIT 368       -- İçinden sadece 368 tane çeker
),
yeni_dagilim AS (
    -- 3. ADIM: Bu seçtiğimiz 368 adet rastgele mahalle verisinin yüzdelerini hesaplıyoruz
    SELECT 
        province,
        district,
        neighborhood,
        COUNT(*) AS yeni_sistem_anket_adedi,
        (COUNT(*) / 368) * 100 AS yeni_sistem_pay_yuzde
    FROM rastgele_orneklem
    GROUP BY province, district, neighborhood
)
-- 4. ADIM: İki sonuç kümesini yan yana getirip farkı görüyoruz
SELECT 
    o.province,
    o.district,
    o.neighborhood,
    o.orijinal_anket_adedi,
    ROUND(o.orijinal_pay_yuzde, 2) AS orijinal_pay_yuzde,
    COALESCE(y.yeni_sistem_anket_adedi, 0) AS yeni_sistem_anket_adedi,
    ROUND(COALESCE(y.yeni_sistem_pay_yuzde, 0), 2) AS yeni_sistem_pay_yuzde,
    ROUND(COALESCE(y.yeni_sistem_pay_yuzde, 0) - o.orijinal_pay_yuzde, 2) AS fark_yuzde_puani
FROM orijinal_dagilim o
LEFT JOIN yeni_dagilim y 
  ON o.province = y.province 
 AND o.district = y.district 
 AND o.neighborhood = y.neighborhood
ORDER BY o.orijinal_anket_adedi DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH orijinal_dagilim AS (
    -- 1. ADIM: Tüm Türkiye'deki mağazaların MAHALLE bazlı orijinal ciro/anket dağılım yüzdelerini hesaplıyoruz
    SELECT 
        store_code,
        province,
        district,
        neighborhood,
        COUNT(spending) AS orijinal_anket_adedi,
        -- Yüzdeyi her mağazanın kendi toplamı içinde hesaplamak için OVER(PARTITION BY store_code) kullanıyoruz
        (COUNT(spending) / SUM(COUNT(spending)) OVER(PARTITION BY store_code)) * 100 AS orijinal_pay_yuzde
    FROM dev_country_turkey.spendingapp
    WHERE date_format(to_timestamp(survey_date, 'M/d/yyyy h:m:ss a'), 'yyyy') = '2025'
      AND store_code IS NOT NULL
    GROUP BY store_code, province, district, neighborhood
),
rastgele_orneklem_sirali AS (
    -- 2. ADIM: Her mağazanın kendi içindeki verileri tamamen karıştırıp (rand) satır numarası veriyoruz
    SELECT 
        store_code,
        province,
        district,
        neighborhood,
        ROW_NUMBER() OVER(PARTITION BY store_code ORDER BY rand()) as rn
    FROM dev_country_turkey.spendingapp
    WHERE date_format(to_timestamp(survey_date, 'M/d/yyyy h:m:ss a'), 'yyyy') = '2025'
      AND store_code IS NOT NULL
),
rastgele_orneklem AS (
    -- 3. ADIM: Her mağaza için karıştırılmış veriden ilk 368 satırı (optimal hedefimiz) kesip alıyoruz
    SELECT 
        store_code,
        province,
        district,
        neighborhood
    FROM rastgele_orneklem_sirali
    WHERE rn <= 368
),
yeni_dagilim AS (
    -- 4. ADIM: Bu seçilen 368'er adet anketin mağaza içi mahalle yüzdelerini hesaplıyoruz
    SELECT 
        store_code,
        province,
        district,
        neighborhood,
        COUNT(*) AS yeni_sistem_anket_adedi,
        (COUNT(*) / 368.0) * 100 AS yeni_sistem_pay_yuzde
    FROM rastgele_orneklem
    GROUP BY store_code, province, district, neighborhood
),
detayli_fark_matrisi AS (
    -- 5. ADIM: Orijinal dağılım ile 368 örneklemli dağılımı MAHAHALLE bazında yan yana getiriyoruz
    SELECT 
        o.store_code,
        o.province,
        o.district,
        o.neighborhood,
        o.orijinal_anket_adedi,
        ROUND(o.orijinal_pay_yuzde, 2) AS orijinal_pay_yuzde,
        COALESCE(y.yeni_sistem_anket_adedi, 0) AS yeni_sistem_anket_adedi,
        ROUND(COALESCE(y.yeni_sistem_pay_yuzde, 0), 2) AS yeni_sistem_pay_yuzde,
        ROUND(COALESCE(y.yeni_sistem_pay_yuzde, 0) - o.orijinal_pay_yuzde, 2) AS fark_yuzde_puani
    FROM orijinal_dagilim o
    LEFT JOIN yeni_dagilim y 
      ON o.store_code = y.store_code
     AND o.province = y.province 
     AND o.district = y.district 
     AND o.neighborhood = y.neighborhood
)
-- 6. ADIM: MAĞAZA BAZLI ÖZET RAPOR
-- Her mağazada en çok sapan mahallenin ne kadar saptığını (hata sınırlarını) ölçüyoruz
SELECT 
    store_code,
    COUNT(DISTINCT CONCAT(province, district, neighborhood)) AS toplam_aktif_mahalle_sayisi,
    SUM(orijinal_anket_adedi) AS 2025_toplam_orijinal_anket,
    MIN(fark_yuzde_puani) AS en_yuksek_negatif_sapma_yuzde_puani,
    MAX(fark_yuzde_puani) AS en_yuksek_pozitif_sapma_yuzde_puani,
    -- Kalite Kontrol: Ortalama mutlak hata ne kadar küçükse model o kadar kusursuzdur
    ROUND(AVG(ABS(fark_yuzde_puani)), 3) AS ortalama_mutlak_sapma
FROM detayli_fark_matrisi
GROUP BY store_code
ORDER BY ortalama_mutlak_sapma ASC;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    store_code,
    date_format(to_timestamp(survey_date, 'M/d/yyyy h:m:ss a'), 'yyyy-MM'),
    count(survey_date)
FROM dev_country_turkey.spendingapp 
WHERE to_date(to_timestamp(survey_date, 'M/d/yyyy h:m:ss a')) >= '2026-01-01'
AND store_code IS NOT NULL
GROUP BY ALL
ORDER BY 1,2